---
## 1. Podejście DuckDB: Leniwa ewaluacja (Lazy Evaluation)

DuckDB pozwala na odroczenie operacji do momentu zażądania wyniku:
1. DuckDB pozwala na stworzenie planu zapytania jeszcze przed rozpoczęciem przetwarzania danych.
2. Wbudowane mechanizmy sprawiają, że zamiast ładowania do pamięci RAM **całej wielkiej tabeli** DuckDB może odczytać wyłącznie kolumny **niezbędne** do realizacji zapytania, jak i odrzucać rekordy niespełniające warunków już na etapie odczytu.
3. Ta metoda pozbywa się również problemu tworzenia przez program pośrednich zbiorów danych.

#### Monitorowanie pamięci
Definiujemy ścieżkę do pliku parquet z używaną przez nas tabelą oraz tworzymy funkcję, która pozwala nam zwrócić aktualne zużycie pamięci RSS - faktycznie zajętej w RAM-ie.

In [1]:
import os
import time
import psutil
import gc
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

parquet_path = "SalesOrderDetail_upsampled_2GB.parquet"

process = psutil.Process(os.getpid())

def get_ram_mb():
    return process.memory_info().rss / (1024 * 1024)

print(f"Plik danych: {parquet_path}")
if os.path.exists(parquet_path):
    print(f"Rozmiar pliku na dysku: {os.path.getsize(parquet_path) / (1024**3):.2f} GB")
print(f"Początkowe zużycie RAM: {get_ram_mb():.1f} MB")

Plik danych: SalesOrderDetail_upsampled_2GB.parquet
Rozmiar pliku na dysku: 1.68 GB
Początkowe zużycie RAM: 154.1 MB


### Pytanie biznesowe:  Jaka jest różnica sprzedaży ilościowej rowerów trekkingowych zależnie od miesiąca?

Interesuje nas jak sezon wpływa na sprzedaż konkretnej podkategorii produktów.

W tym przypadku będziemy porównywać jak różni się ilość sprzedanych, przeznaczonych do zaawansowanej turystyki, rowerów trekkingowych (Touring Bikes, ProductID 953-979) w miesiącach o drastycznie różnym klimacie pogodowym - styczniu i maju 2014 roku.

#### Pierwsze kroki

Aby uzyskać odpowiedź na to pytanie, najpierw tworzymy leniwą relację poprzez wywołanie funkcji **duckdb.read_parquet()**, gdzie DuckDB nie wczytuje żadnych danych, a jedynie sprawdza schemat pliku korzystając z jego metadanych. Kolejnie definiujemy plan naszego zapytania.


In [2]:
ram_before_duck = get_ram_mb()
print(f"[DuckDB] RAM przed zdefiniowaniem relacji: {ram_before_duck:.1f} MB")

rel = duckdb.read_parquet(parquet_path)

lazy_plan = rel \
.filter("ProductID BETWEEN 953 AND 979 " "AND YEAR(ModifiedDate) = 2014 AND MONTH(ModifiedDate) IN (1, 5)") \
.aggregate("Month(ModifiedDate) AS SalesMonth, SUM(OrderQty) AS SoldQty", "Month(ModifiedDate)") \
.order("SalesMonth") 

ram_after_lazy_plan = get_ram_mb()
print(f"[DuckDB] RAM po zdefiniowaniu leniwego planu:  {ram_after_lazy_plan:.1f} MB")
print(f"[DuckDB] Narzut pamięciowy definicji:          {ram_after_lazy_plan - ram_before_duck:+.2f} MB (zero alokacji danych!)")

[DuckDB] RAM przed zdefiniowaniem relacji: 154.1 MB
[DuckDB] RAM po zdefiniowaniu leniwego planu:  179.3 MB
[DuckDB] Narzut pamięciowy definicji:          +25.22 MB (zero alokacji danych!)


#### Dodatek

DuckDB daje nam dostęp do funkcji "wyjaśniających" plan logiczny zapytania oraz przewidywanego wyniku.

#### Uruchomienie zapytania



In [3]:
t0 = time.time()
duckdb_result = lazy_plan.fetchall()
duckdb_time = time.time() - t0
ram_peak_duck = get_ram_mb()

print(f"[DuckDB] Liczba sprzedanych rowerów trekkingowych w styczniu: {duckdb_result[0][1]} ")
print(f"[DuckDB] Liczba sprzedanych rowerów trekkingowych w maju: {duckdb_result[1][1]} ")
print(f"[DuckDB] Czas wykonania:       {duckdb_time:.4f} s")
print(f"[DuckDB] RAM po wykonaniu:     {ram_peak_duck:.1f} MB (wzrost zaledwie o {ram_peak_duck - ram_before_duck:+.1f} MB)")

[DuckDB] Liczba sprzedanych rowerów trekkingowych w styczniu: 404784 
[DuckDB] Liczba sprzedanych rowerów trekkingowych w maju: 620230 
[DuckDB] Czas wykonania:       0.1015 s
[DuckDB] RAM po wykonaniu:     235.6 MB (wzrost zaledwie o +81.5 MB)


--- 
## 2. Podejście Pandas: Chciwa ewaluacja (Eager Evaluation)

W tradycyjnych bibliotekach takich jak Pandas wywołanie `read_parquet` wykonuje się **natychmiastowo i bez wiedzy o kolejnych operacjach** (*Eager Execution*):
1. Pandas musi zaalokować w pamięci RAM **wszystkie 66,724,350 wierszy i wszystkie 11 kolumn**.
2. Skompresowany plik Parquet (~1.8 GB) po rozpakowaniu do struktur DataFrame w RAM zajmuje **od 5 do ponad 10 GB pamięci operacyjnej**.
3. Dopiero po zaalokowaniu gigabajtów danych, Pandas filtruje wiersze i sumuje jedną kolumnę — odrzucając 95% wcześniej załadowanych danych!

In [ ]:
gc.collect()

ram_before_pandas = get_ram_mb()
print(f"[Pandas] RAM przed startem: {ram_before_pandas:.1f} MB")
print("-> Rozpoczynamy chciwe wczytywanie pd.read_parquet()...")
print("-> Obserwuj drastyczny skok alokacji pamięci RAM przez proces!")

t0 = time.time()
pandas_success = False
pandas_time = 0.0
pandas_result = 0.0
ram_peak_pandas = ram_before_pandas

try:
    df = pd.read_parquet(parquet_path)
    ram_after_read = get_ram_mb()
    print(f"-> [Pandas] RAM zaraz po read_parquet: {ram_after_read:.1f} MB (skok o +{ram_after_read - ram_before_pandas:.1f} MB!)")
    
    pandas_filtered = df[(df["ProductID"].between(953, 979)) &
        (df["ModifiedDate"].dt.year == 2014) &
        (df["ModifiedDate"].dt.month.isin([4, 11]))].copy()
    pandas_filtered["SalesMonth"] = pandas_filtered["ModifiedDate"].dt.month
    pandas_result = pandas_filtered.groupby(["SalesMonth"], as_index=False) \
        .agg(TotalQuantitySold=("OrderQty", "sum")) \
        .sort_values(["SalesMonth"])
    pandas_time = time.time() - t0
    ram_peak_pandas = get_ram_mb()
    pandas_success = True

    print(f"[Pandas] Liczba sprzedanych rowerów trekkingowych w styczniu: {int(pandas_result.iloc[0, 1])} ")
    print(f"[Pandas] Liczba sprzedanych rowerów trekkingowych w maju: {int(pandas_result.iloc[1, 1])} ")
    print(f"[Pandas] Czas wykonania:       {pandas_time:.2f} s")
    print(f"[Pandas] Szczytowy RAM:        {ram_peak_pandas:.1f} MB (przyrost o +{ram_peak_pandas - ram_before_pandas:.1f} MB)")
    
    
    del df
    gc.collect()

except MemoryError:
    print("\n[!] AWARIA: Out Of Memory (OOM)!")
    print("Pamięć RAM została wyczerpana przez podejście Eager Evaluation.")

## 3. Dajmy szansę Pandasowi
- wskazmy tylko potrzebne kolumny

In [4]:
gc.collect()
ram_before_pandas = get_ram_mb()
print(f"[Pandas] RAM przed startem: {ram_before_pandas:.1f} MB")
print("-> Rozpoczynamy chciwe wczytywanie pd.read_parquet()...")
print("-> Obserwuj skok alokacji pamięci RAM przez proces!")

t0 = time.time()
pandas_success = False
pandas_time = 0.0
pandas_result = None
ram_peak_pandas = ram_before_pandas

try:
    df = pd.read_parquet(
        parquet_path, 
        columns=["ProductID", "ModifiedDate", "OrderQty"]
    )
    ram_after_read = get_ram_mb()
    print(f"-> [Pandas] RAM zaraz po read_parquet: {ram_after_read:.1f} MB (skok o +{ram_after_read - ram_before_pandas:.1f} MB!)")
    
    mask = (
        df["ProductID"].between(953, 979) &
        (df["ModifiedDate"].dt.year == 2014) &
        (df["ModifiedDate"].dt.month.isin([1, 5]))
    )
    pandas_filtered = df[mask].copy()
    
    pandas_filtered["SalesMonth"] = pandas_filtered["ModifiedDate"].dt.month
    pandas_result = (
        pandas_filtered.groupby(["SalesMonth"], as_index=False)
        .agg(TotalQuantitySold=("OrderQty", "sum"))
        .sort_values(["SalesMonth"])
    )
    
    pandas_time = time.time() - t0
    ram_peak_pandas = get_ram_mb()
    pandas_success = True

    print(f"[Pandas] Liczba sprzedanych rowerów trekkingowych w styczniu: {int(pandas_result.iloc[0]['TotalQuantitySold'])}")
    print(f"[Pandas] Liczba sprzedanych rowerów trekkingowych w maju:     {int(pandas_result.iloc[1]['TotalQuantitySold'])}")
    print(f"[Pandas] Czas wykonania:       {pandas_time:.2f} s")
    print(f"[Pandas] Szczytowy RAM:        {ram_peak_pandas:.1f} MB (przyrost o +{ram_peak_pandas - ram_before_pandas:.1f} MB)")
    
    del df
    del pandas_filtered
    gc.collect()

except MemoryError:
    print("\n[!] AWARIA: Out Of Memory (OOM)!")
    print("Pamięć RAM została wyczerpana przez podejście Eager Evaluation.")

[Pandas] RAM przed startem: 235.8 MB
-> Rozpoczynamy chciwe wczytywanie pd.read_parquet()...
-> Obserwuj skok alokacji pamięci RAM przez proces!
-> [Pandas] RAM zaraz po read_parquet: 2092.9 MB (skok o +1857.1 MB!)
[Pandas] Liczba sprzedanych rowerów trekkingowych w styczniu: 404784
[Pandas] Liczba sprzedanych rowerów trekkingowych w maju:     620230
[Pandas] Czas wykonania:       0.91 s
[Pandas] Szczytowy RAM:        3066.3 MB (przyrost o +2830.6 MB)


--- 
## 4. Wnioski biznesowe (do prezentacji)

1. **Brak potrzeby skalowania w chmurze (Cost Efficiency):**  
   Dzięki leniwej ewaluacji analityk może przetwarzać dziesiątki milionów rekordów bezpośrednio na własnym laptopie, bez konieczności uruchamiania drogich klastrów Sparka czy hurtowni Snowflake/BigQuery.

2. **Eliminacja ryzyka OOM (Business Continuity):**  
   W podejściu Eager (Pandas) wzrost wolumenu danych natychmiast prowadzi do błędu `MemoryError` i przerwania pracy. DuckDB dzięki przetwarzaniu strumieniowemu (Out-of-Core) jest całkowicie odporny na przekroczenie limitu RAM.

3. **Zero-ETL / Zero-Copy:**  
   Nie trzeba budować czasochłonnych potoków ETL ani importować danych do dedykowanych baz – DuckDB czyta pliki Parquet/CSV bezpośrednio w miejscu ich składowania.